In [1]:
import polars as pl

In [2]:
file_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\fhvhv_tripdata_2025-01.parquet"

In [3]:
df_head = pl.scan_parquet(file_path).head(5).collect()

In [4]:
df_head.schema

Schema([('hvfhs_license_num', String),
        ('dispatching_base_num', String),
        ('originating_base_num', String),
        ('request_datetime', Datetime(time_unit='us', time_zone=None)),
        ('on_scene_datetime', Datetime(time_unit='us', time_zone=None)),
        ('pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('PULocationID', Int32),
        ('DOLocationID', Int32),
        ('trip_miles', Float64),
        ('trip_time', Int64),
        ('base_passenger_fare', Float64),
        ('tolls', Float64),
        ('bcf', Float64),
        ('sales_tax', Float64),
        ('congestion_surcharge', Float64),
        ('airport_fee', Float64),
        ('tips', Float64),
        ('driver_pay', Float64),
        ('shared_request_flag', String),
        ('shared_match_flag', String),
        ('access_a_ride_flag', String),
        ('wav_request_flag', String),
        ('wav_match_flag', String),
 

In [5]:
df_head

hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag,cbd_congestion_fee
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],i32,i32,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,f64
"""HV0003""","""B03404""","""B03404""",2025-01-01 00:28:07,2025-01-01 00:31:17,2025-01-01 00:33:25,2025-01-01 00:54:24,148,211,1.32,1259,18.21,0.0,0.46,1.5,2.75,0.0,0.0,15.71,"""N""","""N""","""N""","""N""","""N""",0.0
"""HV0005""","""B03406""",null,2025-01-01 00:18:33,null,2025-01-01 00:29:49,2025-01-01 00:41:52,127,244,3.313,723,16.97,0.0,0.47,1.51,0.0,0.0,0.0,23.57,"""N""","""N""","""N""","""N""","""N""",0.0
"""HV0003""","""B03404""","""B03404""",2025-01-01 00:28:22,2025-01-01 00:31:52,2025-01-01 00:32:39,2025-01-01 01:20:33,132,181,13.43,2874,40.36,0.0,1.18,3.8,0.0,2.5,0.0,47.75,"""N""","""N""","""N""","""N""","""N""",0.0
"""HV0003""","""B03404""","""B03404""",2025-01-01 00:27:13,2025-01-01 00:33:58,2025-01-01 00:34:55,2025-01-01 00:39:19,76,76,0.82,264,12.24,0.0,0.34,1.08,0.0,0.0,0.0,12.2,"""N""","""N""","""N""","""N""","""N""",0.0
"""HV0003""","""B03404""","""B03404""",2025-01-01 00:33:29,2025-01-01 00:45:46,2025-01-01 00:46:19,2025-01-01 00:53:56,76,76,1.61,457,19.76,0.0,0.54,1.76,0.0,0.0,0.0,17.67,"""N""","""N""","""N""","""N""","""N""",0.0


In [6]:
import polars as pl

# Pattern to read all 12 Parquet files lazily
file_pattern = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\fhvhv_tripdata_2025-*.parquet"

print("Starting to process all 12 months...")

# Define lazy processing pipeline
lazy_df = (
    pl.scan_parquet(file_pattern)
    # Select essential columns for ride-hailing analytics
    .select([
        "hvfhs_license_num",
        "request_datetime",
        "pickup_datetime",
        "dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "trip_time",
        "base_passenger_fare",
        "tolls",
        "congestion_surcharge",
        "airport_fee",
        "tips",
        "driver_pay",
        "cbd_congestion_fee"
    ])
    # Add calculated metrics
    .with_columns([
        # Company Name
        pl.when(pl.col("hvfhs_license_num") == "HV0003").then(pl.lit("Uber"))
          .when(pl.col("hvfhs_license_num") == "HV0005").then(pl.lit("Lyft"))
          .otherwise(pl.lit("Other"))
          .alias("company_name"),

        # Wait Time in Minutes (Pickup - Request)
        ((pl.col("pickup_datetime") - pl.col("request_datetime")).dt.total_seconds() / 60.0)
          .alias("wait_time_minutes"),

        # Total Revenue Paid by Passenger
        (
            pl.col("base_passenger_fare") 
            + pl.col("tolls") 
            + pl.col("congestion_surcharge") 
            + pl.col("airport_fee") 
            + pl.col("tips") 
            + pl.col("cbd_congestion_fee").fill_null(0.0)
        ).alias("total_gross_revenue"),

        # Company Margin (Base Fare - Driver Pay)
        (pl.col("base_passenger_fare") - pl.col("driver_pay")).alias("company_margin")
    ])
)

# Output path for the unified parquet dataset
output_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\processed_nyc_ride_hailing_2025.parquet"

# Stream data directly to target parquet file without overloading RAM
lazy_df.sink_parquet(output_path, compression="snappy")

print("All 12 months have been successfully processed, merged, and saved to:")
print(output_path)

Starting to process all 12 months...
All 12 months have been successfully processed, merged, and saved to:
C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\processed_nyc_ride_hailing_2025.parquet


In [6]:
file_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\processed_nyc_ride_hailing_2025.parquet"

In [7]:
# Inspect schema and head (first 5 rows)
df_head = pl.scan_parquet(file_path).head(5).collect()

In [8]:
df_head.schema

Schema([('hvfhs_license_num', String),
        ('request_datetime', Datetime(time_unit='us', time_zone=None)),
        ('pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('PULocationID', Int32),
        ('DOLocationID', Int32),
        ('trip_miles', Float64),
        ('trip_time', Int64),
        ('base_passenger_fare', Float64),
        ('tolls', Float64),
        ('congestion_surcharge', Float64),
        ('airport_fee', Float64),
        ('tips', Float64),
        ('driver_pay', Float64),
        ('cbd_congestion_fee', Float64),
        ('company_name', String),
        ('wait_time_minutes', Float64),
        ('total_gross_revenue', Float64),
        ('company_margin', Float64)])

In [9]:
df_head

hvfhs_license_num,request_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,congestion_surcharge,airport_fee,tips,driver_pay,cbd_congestion_fee,company_name,wait_time_minutes,total_gross_revenue,company_margin
str,datetime[μs],datetime[μs],datetime[μs],i32,i32,f64,i64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64
"""HV0003""",2025-01-01 00:28:07,2025-01-01 00:33:25,2025-01-01 00:54:24,148,211,1.32,1259,18.21,0.0,2.75,0.0,0.0,15.71,0.0,"""Uber""",5.3,20.96,2.5
"""HV0005""",2025-01-01 00:18:33,2025-01-01 00:29:49,2025-01-01 00:41:52,127,244,3.313,723,16.97,0.0,0.0,0.0,0.0,23.57,0.0,"""Lyft""",11.266667,16.97,-6.6
"""HV0003""",2025-01-01 00:28:22,2025-01-01 00:32:39,2025-01-01 01:20:33,132,181,13.43,2874,40.36,0.0,0.0,2.5,0.0,47.75,0.0,"""Uber""",4.283333,42.86,-7.39
"""HV0003""",2025-01-01 00:27:13,2025-01-01 00:34:55,2025-01-01 00:39:19,76,76,0.82,264,12.24,0.0,0.0,0.0,0.0,12.2,0.0,"""Uber""",7.7,12.24,0.04
"""HV0003""",2025-01-01 00:33:29,2025-01-01 00:46:19,2025-01-01 00:53:56,76,76,1.61,457,19.76,0.0,0.0,0.0,0.0,17.67,0.0,"""Uber""",12.833333,19.76,2.09


In [10]:
lazy_df = pl.scan_parquet(file_path)

In [11]:
total_rows = lazy_df.select(pl.len()).collect().item()
total_cols = len(lazy_df.collect_schema().names())

In [12]:
print(f"Total Rows: {total_rows:,}")
print(f"Total Columns: {total_cols}")

Total Rows: 243,589,684
Total Columns: 19


In [13]:
# Scan parquet file lazily and count nulls for each column
null_counts = (
    pl.scan_parquet(file_path)
    .select([pl.col(col).null_count().alias(col) for col in pl.scan_parquet(file_path).collect_schema().names()])
    .collect()
)

print("=== HƏR SÜTUN ÜZRƏ NULL (BOŞ) DƏYƏRLƏRİN SAYI ===")
print(null_counts.transpose(include_header=True, header_name="column_name", column_names=["null_count"]))

=== HƏR SÜTUN ÜZRƏ NULL (BOŞ) DƏYƏRLƏRİN SAYI ===
shape: (19, 2)
┌─────────────────────┬────────────┐
│ column_name         ┆ null_count │
│ ---                 ┆ ---        │
│ str                 ┆ u32        │
╞═════════════════════╪════════════╡
│ hvfhs_license_num   ┆ 0          │
│ request_datetime    ┆ 0          │
│ pickup_datetime     ┆ 0          │
│ dropoff_datetime    ┆ 0          │
│ PULocationID        ┆ 0          │
│ …                   ┆ …          │
│ cbd_congestion_fee  ┆ 0          │
│ company_name        ┆ 0          │
│ wait_time_minutes   ┆ 0          │
│ total_gross_revenue ┆ 0          │
│ company_margin      ┆ 0          │
└─────────────────────┴────────────┘


In [22]:
pip install duckdb

   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/15.4 MB 8.6 MB/s eta 0:00:02
   ----------------- ---------------------- 6.6/15.4 MB 15.8 MB/s eta 0:00:01
   ----------------------------- ---------- 11.3/15.4 MB 18.2 MB/s eta 0:00:01
   ---------------------------------------  15.2/15.4 MB 19.0 MB/s eta 0:00:01
   ---------------------------------------- 15.4/15.4 MB 18.4 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"
con = duckdb.connect(db_path)

print("Creating dim_company table...")

con.execute("""
    CREATE OR REPLACE TABLE dim_company AS
    SELECT 
        company_id,
        CASE 
            WHEN company_id = 'HV0003' THEN 'Uber'
            WHEN company_id = 'HV0005' THEN 'Lyft'
            ELSE 'Other'
        END AS company_name,
        CASE 
            WHEN company_id = 'HV0003' THEN 'Uber Technologies, Inc.'
            WHEN company_id = 'HV0005' THEN 'Lyft, Inc.'
            ELSE 'Unknown'
        END AS company_full_name
    FROM (
        SELECT DISTINCT company_id FROM fact_trips
    )
""")

print("dim_company table successfully created!")

# Verify content
print(con.execute("SELECT * FROM dim_company").df())

con.close()

Creating dim_company table...
dim_company table successfully created!
  company_id company_name        company_full_name
0     HV0003         Uber  Uber Technologies, Inc.
1     HV0005         Lyft               Lyft, Inc.


In [21]:
db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"
output_path = r"C:\Users\ASUS\Desktop\nyc\dim_date.parquet"

con = duckdb.connect(db_path)

print("Recreating dim_date table with pure full_date, day_short, and month_short...")

con.execute(f"""
    COPY (
        WITH date_series AS (
            SELECT CAST(range AS DATE) AS full_date
            FROM range(DATE '2025-01-01', DATE '2026-01-01', INTERVAL '1 day')
        )
        SELECT
            full_date,
            
            -- Year, Month, Day numbers
            CAST(YEAR(full_date) AS SMALLINT) AS year_num,
            CAST(MONTH(full_date) AS UTINYINT) AS month_num,
            CAST(DAY(full_date) AS UTINYINT) AS day_num,
            
            -- Month Names (Full & Short)
            STRFTIME(full_date, '%B') AS month_name,       -- January, February...
            STRFTIME(full_date, '%b') AS month_short,      -- Jan, Feb...
            
            -- Day Names (Full & Short)
            STRFTIME(full_date, '%A') AS day_name,         -- Monday, Tuesday...
            STRFTIME(full_date, '%a') AS day_short,        -- Mon, Tue...
            
            -- Day of week number (1 = Monday, 7 = Sunday)
            CAST(DAYOFWEEK(full_date) AS UTINYINT) AS day_of_week_num,
            
            -- Day Type (Weekday / Weekend)
            CASE 
                WHEN DAYOFWEEK(full_date) IN (6, 7) THEN 'Weekend'
                ELSE 'Weekday'
            END AS day_type,
            
            -- Quarter & Year-Month
            CAST(QUARTER(full_date) AS UTINYINT) AS quarter_num,
            STRFTIME(full_date, '%Y-%m') AS year_month
        FROM date_series
        ORDER BY full_date ASC
    ) TO '{output_path}' (FORMAT PARQUET, COMPRESSION 'ZSTD', COMPRESSION_LEVEL 3);
""")

print("dim_date.parquet successfully recreated at:", output_path)
con.close()

Recreating dim_date table with pure full_date, day_short, and month_short...
dim_date.parquet successfully recreated at: C:\Users\ASUS\Desktop\nyc\dim_date.parquet


In [4]:
csv_path = r"C:\Users\ASUS\Desktop\nyc\taxi_zone_lookup.csv"
db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"

con = duckdb.connect(db_path)

print("Creating dim_location table from official lookup CSV...")

con.execute(f"""
    CREATE OR REPLACE TABLE dim_location AS
    SELECT 
        CAST(LocationID AS INT) AS location_id,
        Borough AS borough,
        Zone AS zone_name,
        service_zone
    FROM read_csv_auto('{csv_path}')
""")

print("dim_location table successfully created!")

# Preview first 5 rows
print(con.execute("SELECT * FROM dim_location LIMIT 5").df())

con.close()

Creating dim_location table from official lookup CSV...
dim_location table successfully created!
   location_id        borough                zone_name service_zone
0            1            EWR           Newark Airport          EWR
1            2         Queens              Jamaica Bay    Boro Zone
2            3          Bronx  Allerton/Pelham Gardens    Boro Zone
3            4      Manhattan            Alphabet City  Yellow Zone
4            5  Staten Island            Arden Heights    Boro Zone


In [5]:
import os

db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"
output_dir = r"C:\Users\ASUS\Desktop\nyc"

con = duckdb.connect(db_path)

print("Exporting dimension tables to Parquet files...")

# 1. Export dim_company
con.execute(f"""
    COPY dim_company 
    TO '{os.path.join(output_dir, "dim_company.parquet")}' 
    (FORMAT PARQUET, COMPRESSION 'SNAPPY');
""")
print("- dim_company.parquet exported successfully!")

# 2. Export dim_date
con.execute(f"""
    COPY dim_date 
    TO '{os.path.join(output_dir, "dim_date.parquet")}' 
    (FORMAT PARQUET, COMPRESSION 'SNAPPY');
""")
print("- dim_date.parquet exported successfully!")

# 3. Export dim_location
con.execute(f"""
    COPY dim_location 
    TO '{os.path.join(output_dir, "dim_location.parquet")}' 
    (FORMAT PARQUET, COMPRESSION 'SNAPPY');
""")
print("- dim_location.parquet exported successfully!")

con.close()
print("\nAll dimension tables successfully saved in:", output_dir)

Exporting dimension tables to Parquet files...
- dim_company.parquet exported successfully!
- dim_date.parquet exported successfully!
- dim_location.parquet exported successfully!

All dimension tables successfully saved in: C:\Users\ASUS\Desktop\nyc


In [20]:
import duckdb

db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"
output_path = r"C:\Users\ASUS\Desktop\nyc\fact_trips.parquet"

con = duckdb.connect(db_path)

print("Exporting fully optimized and downcasted fact table...")

con.execute(f"""
    COPY (
        SELECT
            -- Date dimension linkage
            CAST(pickup_datetime AS DATE) AS pickup_date,
            
            -- Time bucket (1 byte integer)
            CAST(HOUR(pickup_datetime) AS UTINYINT) AS pickup_hour,
            
            -- Time window categorization
            CASE 
                WHEN HOUR(pickup_datetime) >= 6 AND HOUR(pickup_datetime) < 12 THEN '1. Morning'
                WHEN HOUR(pickup_datetime) >= 12 AND HOUR(pickup_datetime) < 18 THEN '2. Afternoon'
                WHEN HOUR(pickup_datetime) >= 18 AND HOUR(pickup_datetime) < 24 THEN '3. Evening'
                ELSE '4. Night'
            END AS time_window,
            
            -- Dimensions
            CAST(company_id AS VARCHAR) AS company_id,
            CAST(PULocationID AS SMALLINT) AS PULocationID,
            CAST(DOLocationID AS SMALLINT) AS DOLocationID,
            
            -- Volume Metrics
            CAST(COUNT(*) AS INTEGER) AS total_trips,
            
            -- Duration & Distance Metrics
            CAST(AVG(epoch(pickup_datetime - request_datetime) / 60.0) AS FLOAT) AS avg_wait_time_minutes,
            CAST(SUM(epoch(dropoff_datetime - pickup_datetime) / 60.0) AS FLOAT) AS total_trip_time_minutes,
            CAST(SUM(trip_miles) AS FLOAT) AS total_trip_miles,
            
            -- Financial Metrics
            CAST(SUM(base_passenger_fare) AS FLOAT) AS total_base_fare,
            CAST(SUM(tolls) AS FLOAT) AS total_tolls,
            CAST(SUM(congestion_surcharge) AS FLOAT) AS total_congestion_surcharge,
            CAST(SUM(airport_fee) AS FLOAT) AS total_airport_fee,
            CAST(SUM(tips) AS FLOAT) AS total_tips,
            CAST(SUM(driver_pay) AS FLOAT) AS total_driver_pay,
            CAST(SUM(cbd_congestion_fee) AS FLOAT) AS total_cbd_congestion_fee,
            CAST(SUM(total_gross_revenue) AS FLOAT) AS total_gross_revenue,
            CAST(SUM(company_margin) AS FLOAT) AS total_company_margin
        FROM fact_trips
        GROUP BY ALL
    ) TO '{output_path}' (FORMAT PARQUET, COMPRESSION 'ZSTD', COMPRESSION_LEVEL 3);
""")

print("fact_trips.parquet successfully created at:", output_path)
con.close()

Exporting fully optimized and downcasted fact table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_trips.parquet successfully created at: C:\Users\ASUS\Desktop\nyc\fact_trips.parquet


In [6]:
db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"
output_parquet = r"C:\Users\ASUS\Desktop\nyc\fact_trips.parquet"

con = duckdb.connect(db_path)

print("Exporting fact_trips table to Parquet format...")

con.execute(f"""
    COPY fact_trips 
    TO '{output_parquet}' 
    (FORMAT PARQUET, COMPRESSION 'SNAPPY');
""")

print("fact_trips.parquet successfully created at:", output_parquet)
con.close()

Exporting fact_trips table to Parquet format...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_trips.parquet successfully created at: C:\Users\ASUS\Desktop\nyc\fact_trips.parquet
